### py4cytoscape

https://py4cytoscape.readthedocs.io/en/latest/

In [16]:
from pathlib import Path
from rdflib import Graph, RDF, Namespace
import networkx as nx
import py4cytoscape as p4c
import inspect

In [17]:
print(inspect.signature(p4c.set_node_shape_mapping))
print(inspect.signature(p4c.set_node_color_mapping))

(table_column, table_column_values=None, shapes=None, default_shape=None, style_name=None, network=None, base_url='http://127.0.0.1:1234/v1')
(table_column, table_column_values=None, colors=None, mapping_type='c', default_color=None, style_name=None, network=None, base_url='http://127.0.0.1:1234/v1')


### Cytoscape 

/home/flavio/Cytoscape_v3.10.2

cd /media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/cytoscape
wget https://github.com/cytoscape/cytoscape/releases/download/3.10.2/Cytoscape_3_10_2_unix.sh

# make executable
chmod +x Cytoscape_3_10_2_unix.sh

# java 
sudo apt update
sudo apt install openjdk-17-jdk
java -version
whereis java
readlink -f /usr/bin/java
java -version

# openjdk 17.0.18 2026-01-20
# OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-124.04.1)
# OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-124.04.1, mixed mode, sharing)

# install
./Cytoscape_3_10_2_unix.sh

# run
export INSTALL4J_JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
~/Cytoscape_v3.10.2/Cytoscape &


In [8]:
root_owl = Path('../owl/')
BP = Namespace("http://www.biopax.org/release/biopax-level3.owl#")

G = nx.DiGraph()

classes = [
    BP.Pathway,
    BP.BiochemicalReaction,
    BP.Catalysis,
    BP.Control,
    BP.Protein,
    BP.Complex,
    BP.SmallMolecule,
    BP.PhysicalEntity,
]

relations = [
    BP.pathwayComponent,
    BP.left,
    BP.right,
    BP.controller,
    BP.controlled,
    BP.participant,
    BP.component,
]


def read_owl(owl_file):
    fnameowl = root_owl / owl_file

    rdf = Graph()
    rdf.parse(fnameowl, format="xml")

    return rdf

def short(x):
    return str(x).split("#")[-1].split("/")[-1]

def get_name(x, rdf):
    for prop in [BP.displayName, BP.standardName, BP.name]:
        value = next(rdf.objects(x, prop), None)
        if value:
            return str(value)
    return short(x)

In [9]:
owl_file = "R-HSA-165159_level3.owl"
owl_file = "R-HSA-114608_level3.owl"

rdf = read_owl(owl_file)

In [10]:
for cls in classes:
    for node in rdf.subjects(RDF.type, cls):
        node_id = short(node)
        G.add_node(node_id, label=get_name(node, rdf), biopax_type=short(cls))

In [11]:
for rel in relations:
    for s, o in rdf.subject_objects(rel):
        s_id = short(s)
        o_id = short(o)

        if s_id in G.nodes and o_id in G.nodes:
            G.add_edge(s_id, o_id, interaction=short(rel))

In [13]:
p4c.cytoscape_ping()

p4c.create_network_from_networkx(G, title=f"Reactome {owl_file}", collection="Reactome OWL")

You are connected to Cytoscape!
Applying default style...
Applying preferred layout


220

In [19]:
style_name = "Reactome_BioPAX_Style"

p4c.create_visual_style(style_name)

p4c.set_node_label_mapping("label", style_name=style_name)

p4c.set_node_color_mapping(
    "biopax_type",
    ["Protein", "Complex", "SmallMolecule", "BiochemicalReaction", "Pathway"],
    ["#8ecae6", "#ffb703", "#90be6d", "#f94144", "#cdb4db"],
    mapping_type="d",
    default_color="#cccccc",
    style_name=style_name
)

p4c.set_node_shape_mapping(
    "biopax_type",
    ["Protein", "Complex", "SmallMolecule", "BiochemicalReaction", "Pathway"],
    ["ELLIPSE", "ROUND_RECTANGLE", "DIAMOND", "HEXAGON", "RECTANGLE"],
    default_shape="ELLIPSE",
    style_name=style_name
)

p4c.set_visual_style(style_name)
p4c.layout_network("force-directed")

{}

```Python
G.nodes[gene_id]["log2FC"] = log2fc
G.nodes[gene_id]["FDR"] = fdr
G.nodes[gene_id]["mutated"] = True
```